# VaxiMère-QA-CG — Pipeline complet (Google Colab)

Construction du dataset d'intentions multilingue (FR / lingala / kituba) pour la
vaccination pédiatrique au Congo-Brazzaville.

**Étapes** : 1) install 2) vérif GPU 3) récupération du code (clone auto)
4) test rapide (`dryrun`, sans modèle) 5) exécution complète 6) inspection 7) téléchargement.

> Exécutez les cellules **dans l'ordre**. La cellule 3 clone automatiquement le dépôt
> depuis GitHub ; aucune action manuelle n'est nécessaire.

In [ ]:
# 1) Installation des dépendances (Colab fournit déjà torch/transformers)
!pip install -q datasets transformers pandas accelerate sentencepiece huggingface_hub

In [ ]:
# 2) Vérification du GPU (un T4 suffit largement)
!nvidia-smi -L
import torch
print("CUDA disponible :", torch.cuda.is_available())
print("Device :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
# 3) Récupération du code — clone AUTOMATIQUE (avec fallback)
#    Si le dépôt est déjà présent (ou les fichiers uploadés à la racine), on le réutilise.
import os, subprocess
from pathlib import Path
from IPython import get_ipython

REPO_URL = "https://github.com/maick-code/AIMS-Capstone.git"
BRANCH   = "arena/01a03c0c-aims-capstone"   # branche qui contient le pipeline
REPO_DIR = Path("/content/AIMS-Capstone")

def find_repo():
    for c in (REPO_DIR, Path("/content"), Path.cwd()):
        if (c / "run_pipeline.py").exists() and (c / "vaximere").exists():
            return c
    return None

repo = find_repo()
if repo is None:
    # essai 1 : branche du pipeline ; essai 2 : branche par défaut
    for args in (["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)],
                 ["git", "clone", REPO_URL, str(REPO_DIR)]):
        r = subprocess.run(args, capture_output=True, text=True)
        if r.returncode == 0:
            break
    repo = find_repo()

if repo is None:
    print("⚠️ Échec du clonage automatique.")
    print("👉 Uploadez manuellement dans Colab : run_pipeline.py, vaximere/,")
    print("   selftest.py et DATA_CARD.md, puis relancez cette cellule.")
    repo = Path.cwd()
else:
    get_ipython().run_line_magic("cd", str(repo))
    print("✅ Dépôt prêt :", repo)

print("Répertoire de travail :", os.getcwd())
print("Fichiers :", sorted(f for f in os.listdir(repo) if not f.startswith('.')))

In [ ]:
# 4) Test rapide SANS modèle ni réseau (valide tout le câblage du pipeline)
#    Sorties : data/dryrun/ (aucun téléchargement de modèle)
import os
if os.path.exists("/content/AIMS-Capstone/run_pipeline.py"):
    os.chdir("/content/AIMS-Capstone")

!python selftest.py
!python run_pipeline.py --mode dryrun

In [ ]:
# 5) Exécution COMPLÈTE (télécharge mDeBERTa-v3 + NLLB-600M, exécute le pipeline)
#    Durée estimée sur T4 : ~10-20 min selon la connexion.
#    Sorties : data/final/
import os
if os.path.exists("/content/AIMS-Capstone/run_pipeline.py"):
    os.chdir("/content/AIMS-Capstone")

!python run_pipeline.py --mode full

In [ ]:
# 6) Inspection des sorties (data/final/ après `--mode full`, data/dryrun/ après le test)
import json
from pathlib import Path

for base in ("data/final", "data/dryrun"):
    d = Path(base)
    if not d.exists():
        continue
    print(f"== {base} ==")
    for f in sorted(d.glob("*.jsonl")):
        n = sum(1 for _ in f.open(encoding="utf-8"))
        first = next(f.open(encoding="utf-8"), "").strip()
        print(f"  {f.name}: {n} lignes")
        print(f"    aperçu -> {first[:180]}")
    stats = d / "stats_report.json"
    if stats.exists():
        s = json.load(stats.open(encoding="utf-8"))
        print("  stats_report.json :")
        print(json.dumps(s, ensure_ascii=False, indent=2)[:1500])

In [ ]:
# 7) Téléchargement des livrables (DATA_CARD.md est à la racine, pas dans data/)
from google.colab import files
from pathlib import Path

targets = [
    Path("data/final/vaximere_qa_cg_train.jsonl"),
    Path("data/final/faq_validee.json"),
    Path("data/final/stats_report.json"),
    Path("data/final/vaximere_qa_cg_train.csv"),
    Path("DATA_CARD.md"),
]
found = False
for p in targets:
    if p.exists():
        files.download(str(p))
        found = True
    else:
        print("(absent)", p)
if not found:
    print("Aucun livrable trouvé : lancez d'abord `--mode full` (cellule 5).")